# CS570 Team 14 — Annotation Helper

This notebook is for **annotators only**. It does not train any model.

It will:
1. Connect to Google Drive (where the Rico dataset lives)
2. Extract screenshots to local Colab storage so they load fast
3. Show you each screen's image + the list of nodes you need to label

**Run cells top to bottom, one at a time. Wait for each cell to finish before running the next.**

---

## Before you start

- Read `annotation_guidelines.md` first — it explains what each label means
- Have your template CSV open (`templates/annotations_A.csv` or `annotations_B.csv`)
- You do NOT need a GPU — leave runtime as CPU

---
## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Drive mount failed — try running this cell again')

---
## Step 2 — Extract Rico screenshots

This copies the Rico archive from Drive to local Colab storage and extracts it.
Takes **5–10 minutes** the first time each session. Subsequent sessions re-extract because local storage is wiped when Colab disconnects — but the archive stays safe on Drive.

If you already ran this cell earlier in this session and it succeeded, you can skip it.

In [ ]:
import os, glob

DRIVE_ROOT      = '/content/drive/MyDrive/cs570-project'
RICO_TAR_DRIVE  = f'{DRIVE_ROOT}/unique_uis.tar.gz'
RICO_RAW_DIR    = '/content/rico_raw'
RICO_TAR_LOCAL  = '/content/unique_uis.tar.gz'

os.makedirs(RICO_RAW_DIR, exist_ok=True)

# Check if already extracted
existing_pngs = glob.glob(f'{RICO_RAW_DIR}/**/*.png', recursive=True)
if len(existing_pngs) > 1000:
    print(f'✅ Already extracted ({len(existing_pngs):,} screenshots available) — skipping')
elif not os.path.exists(RICO_TAR_DRIVE):
    print(f'❌ Rico archive not found at: {RICO_TAR_DRIVE}')
    print('   Ask Nadia to share the Drive folder with you.')
else:
    print(f'Copying archive from Drive to local storage...')
    os.system(f'cp "{RICO_TAR_DRIVE}" "{RICO_TAR_LOCAL}"')
    print('✅ Copied')

    print('Extracting screenshots (5–10 minutes)...')
    ret = os.system(f'tar -xzf "{RICO_TAR_LOCAL}" -C "{RICO_RAW_DIR}"')
    os.remove(RICO_TAR_LOCAL)  # free up local space

    pngs = glob.glob(f'{RICO_RAW_DIR}/**/*.png', recursive=True)
    if pngs:
        print(f'✅ Extracted {len(pngs):,} screenshots')
        print(f'   Sample: {pngs[0]}')
    else:
        print('❌ Extraction finished but no PNGs found — check the archive')

---
## Step 3 — Verify a single screenshot loads

Run this to confirm everything is working before you start annotating.
You should see a phone screenshot appear below the cell.

In [ ]:
import glob, os
from IPython.display import Image, display

RICO_RAW_DIR = '/content/rico_raw'

sample_pngs = glob.glob(f'{RICO_RAW_DIR}/**/*.png', recursive=True)
if sample_pngs:
    test_img = sample_pngs[0]
    screen_id = os.path.splitext(os.path.basename(test_img))[0]
    print(f'Showing screen_id: {screen_id}')
    display(Image(test_img, width=350))
    print('✅ Images are working — proceed to Step 4')
else:
    print('❌ No PNG files found. Run Step 2 first.')

---
## Step 4 — Enter your assigned screen IDs

Paste in the screen IDs from your assignment (`task_allocation.md`).
Run this cell once, then use the browsing cells below.

In [ ]:
# ── EDIT THIS LIST with your assigned screen IDs ──────────────────────────
# Copy them from task_allocation.md under your name

MY_SCREEN_IDS = [
    # Overlap screens (annotate these independently — do not discuss labels yet)
    '20353', '21089', '23456', '25871', '30124',
    '33567', '40892', '45231', '51678', '58903',
    # Your unique screens (replace with your actual IDs from task_allocation.md)
    # --- Annotator A: paste your 20 IDs below ---
    # '61234', '62891', ...
    # --- Annotator B: paste your 20 IDs below ---
    # '11023', '12456', ...
]

MY_INITIALS = 'na'   # change to your initials — must match what you use in the CSV

# ── Paths (don't change these) ─────────────────────────────────────────────
RICO_RAW_DIR = '/content/rico_raw'
DRIVE_ROOT   = '/content/drive/MyDrive/cs570-project'
PROCESSED_DIR = f'{DRIVE_ROOT}/data/processed'

print(f'Loaded {len(MY_SCREEN_IDS)} screen IDs for annotator: {MY_INITIALS}')
print('Screen IDs:', MY_SCREEN_IDS)

---
## Step 5 — Browse screens and annotate

Run the cell below. It will show you each screen in order:
- The screenshot on the left
- The node list on the right (node_id, class, text, depth, heuristic label)

For each node, decide the label and fill in your CSV.

Change `SCREEN_INDEX` to jump to a different screen (0 = first, 1 = second, etc.).

In [ ]:
import os, glob, json
from IPython.display import Image, display, HTML
import IPython.display as ipd

# ── Change this to browse a different screen ───────────────────────────────
SCREEN_INDEX = 0   # 0 = first screen, 1 = second, ...
# ──────────────────────────────────────────────────────────────────────────

LABEL_NAMES = {0: '🔴 Canonical', 1: '🟡 Translatable', 2: '🟢 Open', -1: '— excluded'}

if SCREEN_INDEX >= len(MY_SCREEN_IDS):
    print(f'❌ SCREEN_INDEX {SCREEN_INDEX} is out of range. Max is {len(MY_SCREEN_IDS) - 1}.')
else:
    screen_id = MY_SCREEN_IDS[SCREEN_INDEX]
    progress  = f'Screen {SCREEN_INDEX + 1} of {len(MY_SCREEN_IDS)}'

    print(f'{'='*60}')
    print(f'  {progress}  |  screen_id: {screen_id}')
    print(f'{'='*60}')

    # ── Show screenshot ────────────────────────────────────────────────────
    img_path = f'{RICO_RAW_DIR}/combined/{screen_id}.png'
    if os.path.exists(img_path):
        print(f'Screenshot: {img_path}')
        display(Image(img_path, width=380))
    else:
        print(f'⚠️  Screenshot not found: {img_path}')
        print('   (The image extraction may be incomplete. Try re-running Step 2.)')

    # ── Show node list ─────────────────────────────────────────────────────
    # First try loading from preprocessed .pt graph (has heuristic labels)
    import torch
    pt_files = glob.glob(f'{PROCESSED_DIR}/**/{screen_id}.pt', recursive=True)

    if pt_files:
        g = torch.load(pt_files[0], weights_only=False)
        print(f'\nNode list (from preprocessed graph — {g["num_nodes"]} nodes):')
        print(f'  App: {g["app_id"]}  |  Label mode: {g.get("label_mode", "unknown")}')
        print()
        print(f'  {"node_id":>8}  {"heuristic":>15}  {"depth":>5}  text/content')
        print(f'  {"-"*70}')
        for i, (nid, lbl) in enumerate(zip(g['node_ids'], g['y'].tolist())):
            if lbl == -1:
                continue  # skip excluded nodes — don't annotate these
            label_str = LABEL_NAMES.get(lbl, str(lbl))
            print(f'  {str(nid):>8}  {label_str:>15}')
        print()
        print('  Excluded nodes (label=-1) are not shown — do not add them to your CSV.')

    else:
        # Fall back to raw JSON if .pt not available
        json_files = glob.glob(f'{RICO_RAW_DIR}/**/{screen_id}.json', recursive=True)
        if json_files:
            with open(json_files[0]) as f:
                raw = json.load(f)

            print(f'\nNode list (from raw JSON hierarchy — no heuristic labels available yet):')
            print(f'  Assign node_ids in DFS traversal order starting at 0.')
            print()

            def walk(node, depth=0, counter=[0]):
                nid = counter[0]
                counter[0] += 1
                cls  = node.get('class', '')[-30:]
                text = (node.get('text') or node.get('content-desc') or '').strip()[:50]
                if text or cls:
                    print(f'  {nid:>8}  depth={depth}  [{cls}]  "{text}"')
                for child in node.get('children', []):
                    walk(child, depth + 1, counter)

            walk(raw)
            print()
            print('  ⚠️  Preprocessing not done yet — node_ids above are estimated.')
            print('     Confirm with Nadia before submitting, or wait for .pt files.')
        else:
            print(f'\n❌ Neither .pt graph nor raw JSON found for screen_id={screen_id}')
            print('   The screen may not be in the dataset. Flag it in your notes.')

    print(f'\n── CSV rows to fill in for this screen ──')
    print(f'  screen_id={screen_id}, node_id=<each node above>, label=<your label>, annotator_id={MY_INITIALS}, notes=')
    print(f'\n  Next screen: set SCREEN_INDEX = {SCREEN_INDEX + 1} and re-run this cell')

---
## Step 6 — (Optional) View a specific screen by ID

Use this if you want to jump directly to a screen you already know the ID of.

In [ ]:
# ── Enter a specific screen ID here ────────────────────────────────────────
JUMP_TO_SCREEN_ID = '20353'
# ──────────────────────────────────────────────────────────────────────────

from IPython.display import Image, display
import os

RICO_RAW_DIR = '/content/rico_raw'
img_path = f'{RICO_RAW_DIR}/combined/{JUMP_TO_SCREEN_ID}.png'

if os.path.exists(img_path):
    print(f'screen_id: {JUMP_TO_SCREEN_ID}')
    display(Image(img_path, width=380))
else:
    print(f'❌ Not found: {img_path}')
    print('   Run Step 2 (extraction) if you have not done so.')

---
## Step 7 — View all your screens as a gallery

Shows thumbnails of all your assigned screens at once — useful for getting a feel for the variety before diving in.

In [ ]:
from IPython.display import Image, display, HTML
import os

RICO_RAW_DIR = '/content/rico_raw'

html_parts = ['<div style="display:flex; flex-wrap:wrap; gap:12px;">']
missing = []

for sid in MY_SCREEN_IDS:
    img_path = f'{RICO_RAW_DIR}/combined/{sid}.png'
    if os.path.exists(img_path):
        import base64
        with open(img_path, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        html_parts.append(
            f'<div style="text-align:center">'
            f'<img src="data:image/png;base64,{b64}" width="140" style="border:1px solid #ccc"/>'
            f'<br><small>{sid}</small>'
            f'</div>'
        )
    else:
        missing.append(sid)
        html_parts.append(
            f'<div style="text-align:center; width:140px; height:200px; border:1px solid #f66; '
            f'display:flex; align-items:center; justify-content:center; font-size:11px; color:#f66">'
            f'MISSING<br>{sid}'
            f'</div>'
        )

html_parts.append('</div>')
display(HTML(''.join(html_parts)))

print(f'\n{len(MY_SCREEN_IDS) - len(missing)} screenshots found, {len(missing)} missing')
if missing:
    print(f'Missing IDs: {missing}')
    print('Flag missing screens in your CSV notes and let Nadia know.')

---
## Quick label reference

| Label | Code | Use when... |
|-------|------|-------------|
| Canonical | 0 | Price, balance, account number, payment button, credential, CAPTCHA |
| Translatable | 1 | Nav label, page title, button for non-financial action, field label, status message |
| Open | 2 | Product photo, promotional banner, decorative image, app tagline |

**When in doubt:** Canonical beats Translatable. Translatable beats Open. Write a note.

**CSV format:**
```
screen_id,node_id,label,annotator_id,notes
20353,2,canonical,na,price shown at checkout total
20353,5,translatable,na,section header
20353,8,open,na,product image
```

See `annotation_guidelines.md` for the full decision flowchart and edge cases.